# 04 — YOLO Litter Detection: Optimised (Task 2 ablation)

**Project:** CNN-Based Aerial Litter Detection for Sustainable Trail and Environmental Cleanup  
**Module:** ST7088CEM Artificial Neural Networks

Task 2, optimisation. Starting from the 640 px baseline (notebook 03), three
modifications are added **cumulatively** and each one's contribution is
quantified against the baseline:

1. **+resolution** — train at 1024 px instead of 640 px.
2. **+augmentation** — domain-motivated augmentation (vertical flips + rotation
   for nadir aerial views, stronger scale jitter, copy-paste, brightness).
3. **+SAHI** — sliced inference at test time (detect on overlapping windows,
   then merge) to recover small objects.

Metrics: mAP@0.5, mAP@0.5:0.95, precision, recall.

## 1. Environment setup (Kaggle GPU + Internet)

In [ ]:
# On Kaggle (Internet enabled), uncomment:
# !git clone -b feature/yolo-optimization https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# import os; os.makedirs('data', exist_ok=True)
# !ls /kaggle/input
# !ln -s /kaggle/input/<dataset-slug>/data/images data/images
# !ln -s /kaggle/input/<dataset-slug>/data/annotations data/annotations
%pip install -q ultralytics sahi pycocotools

import os, sys, platform
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

import torch, ultralytics, sahi
print('Machine:', platform.node(), '|', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('Ultralytics:', ultralytics.__version__, '| SAHI:', sahi.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. YOLO dataset (reused from Phase 1 splits)

In [ ]:
if not Path('data/processed/splits.json').exists():
    !python -m src.data.splits
if not Path('data/yolo/data.yaml').exists():
    !python -m src.data.coco_to_yolo

## 3. Train the optimisation arms

Two fine-tunes at 1024 px: one with standard augmentation (**+resolution**) and
one with the enhanced augmentation block in `configs/yolo_optimized.yaml`
(**+augmentation**). Each is slower than the 640 px baseline.

In [ ]:
!python -m src.training.train_yolo --config configs/yolo_res1024.yaml
!python -m src.training.train_yolo --config configs/yolo_optimized.yaml

## 4. Evaluate each arm on the test split

The two 1024 px models are scored with Ultralytics; the enhanced model is then
scored with SAHI sliced inference (and, as a control, on full images through the
same pycocotools pipeline so SAHI's contribution can be isolated).

In [ ]:
!python -m src.evaluation.evaluate_yolo --config configs/yolo_res1024.yaml --tag "+resolution"
!python -m src.evaluation.evaluate_yolo --config configs/yolo_optimized.yaml --tag "+augmentation"
!python -m src.evaluation.sahi_eval --config configs/yolo_optimized.yaml
!python -m src.evaluation.sahi_eval --config configs/yolo_optimized.yaml --no-slice

## 5. Consolidated ablation

Combines the baseline (carried over from notebook 03) with the three arms.

In [ ]:
!python -m src.evaluation.ablation

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

print('Task 2 ablation:')
display(pd.read_csv('results/tables/yolo_ablation.csv'))
display(IPImage('results/figures/yolo_ablation.png', width=820))

## 6. Findings for the report

- The ablation combines a cumulative **training** progression (baseline →
  +resolution → +augmentation, scored with Ultralytics) with a separate
  **test-time SAHI** experiment (scored with pycocotools). The two eval
  backends are labelled and separated in the figure so they aren't compared
  directly by eye.
- **Higher resolution is the single biggest win.** Going 640 → 1024 px lifts
  mAP@0.5 from 0.793 to 0.858 and mAP@0.5:0.95 from 0.447 to 0.504 — expected,
  since ≈99% of objects occupy <1% of the frame (notebook 01) and 1024 px keeps
  more pixels on each object.
- **Enhanced augmentation adds a small further gain** (mAP@0.5 0.858 → 0.863),
  with a precision/recall trade (higher precision, slightly lower recall). The
  flips/rotation are valid because nadir aerial imagery has no canonical
  orientation — a domain-specific modification.
- **SAHI sliced inference did *not* help here — a notable negative result.**
  Compared fairly against the same model on full images through the same
  pycocotools pipeline, slicing *reduced* mAP@0.5 (0.821 → 0.623) and collapsed
  precision (0.69 → 0.12) while only marginally changing recall. High recall
  confirms the sliced boxes are correctly located (no coordinate bug); the
  problem is a flood of false positives. At 1024 px the detector already
  resolves most litter, and 512 px slices magnify ambiguous ground textures
  (leaves, debris, gravel) into litter-like false detections. SAHI helps when
  the base inference resolution is too small for the objects; on UAVVaste at
  1024 px that condition no longer holds.
- Higher resolution required a smaller batch (8 vs 16) to fit GPU memory —
  a documented practical constraint.